# 🫀 퀘스트 46 · Q7-P — **`p_safe`: P 창이 읽는 게 P 파인가, 직전 T 인가**

| | **MedKOS / `notebooks/quest46_q7p_p_safe.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0063`(Q7-N) · `ailab-2026-0062`(Q7-M) · `ailab-2026-0057`(Q7-F) |
| 규약 | **R22 · R25 · R26 ② · R27 ② ③ · R28 ① ② ③ · R29 ① ② ③ ④ · R30 ① ② ③ · R31 ① ② ③ ④ ⑤** |
| 학습 | **0회** — `svdb_data5.npz` 의 파형만 읽는다 · GPU 불필요 |

## 이거 하나가 P 축의 생사를 정한다

Q7-N 에서 **동시에 두 가지가 나왔다**(R30 ④ — 배타로 읽지 않는다):

- **폭을 맞추니 P 가 이겼다** — `p_full − stt`(85 vs 85) **+0.0745** [+0.0387, +0.1112]
- **그런데 P 창 안에서 이른 쪽이 더 세다** — `p_early` 0.2426 > `p_late` 0.2116 (둘 다 폭 32)

「앞쪽 창이 이긴다」와 「P 파 고유 신호는 없다」가 동시에 성립하는 유일한 설명은
**앞쪽 창이 담는 게 P 파가 아니라 직전 T** 라는 것이다.

```
현재 R = index 100 · 360Hz · 중앙 RR ≈ 282샘플
직전 R = 100 − pre                직전 T ≈ [100 − 0.85·pre,  100 − 0.45·pre]
p_full = index 0–85               p_early = 0–32        p_late = 53–85
```

`pre` 가 **222샘플(617ms · 97bpm)** 아래로 내려가면 직전 T 의 끝이 index 0 을 넘어
**`p_full`·`p_early` 창 안으로 들어온다.** S 는 정의상 조기라 `pre` 가 짧다 →
**창의 내용물 자체가 라벨과 상관된 조기성 누출 경로**다.

⚠️ **이건 RR 특징에 대한 잔차화로는 못 지운다** — 점수의 선형 성분이 아니라
**창의 내용물**이라서다. 그래서 Q7-N 의 통제(lin·quad·hist·rank·층화)를 아무리 조여도
남는다. 실제로 관측이 이 가설과 **전부** 일치한다:

| 예측 | Q7-N 관측 |
|---|---|
| 창이 넓을수록 침입↑ → 차이↑ | 폭 85 **+0.0745** > 폭 32 +0.0310 |
| R 에서 멀수록 침입↑ | `p_early` **+0.2426** > `p_late` +0.2116 |
| 조기성 통제가 강할수록 차이↓ | lin .0745 → hist .0621 → quad .0564 → rank .0466 → 층화 **.0359** |

**이 실험은 그 가설을 「특징」이 아니라 「표본」으로 친다** — 침입이 **기하학적으로
불가능한 비트만** 남기고 헤드라인을 다시 잰다.

## `STT` 에는 있고 `P` 에는 없던 것

Q7-K 는 `STT`(130–215) 에 대해 **다음 QRS 침입**을 막은 `stt_safe`(130–175)를 이미 갖고
있었다. **P 쪽에는 그 짝이 없다.** 이 노트북이 그걸 만든다.

```
p_safe(seg) :  직전 T 끝  100 − 0.45·pre  ≤  seg 시작 − 여유
q_safe(seg) :  다음 QRS   100 + post      ≥  seg 끝   + 여유
```

★ **두 마스크를 모든 창에 걸어 하나의 `SAFE` 코호트를 만든다.** 한쪽 팔만 청소하면
비교가 깨진다 — 그게 Q7-M 이 폭에서 저지른 실수(R27 ③)의 표본 판본이다.

## 사전등록 — 관문

| 관문 | 내용 | 판정 | **지지의 의미** |
|---|---|---|---|
| **P1** ★★ | **`SAFE` 코호트**에서 `p_full − stt`(폭 85) | 등가·우월성 **둘 다** · Bonf 3 | 살아남으면 **P 주장이 훨씬 단단해진다** |
| **P2** ★ | **`SAFE` 코호트**에서 `p_early − p_late`(둘 다 폭 32) | 등가·우월성 둘 다 · Bonf 3 | 무너지면 §4 의 사후 관찰이 **침입이었다** |
| **P3** ★ | rho(개체별 T중첩 격차, 개체별 `p_early` 초과) | 하한 > 0 · Bonf 3 | ⚠️ **지지 = 교란 있음**(Q7-F F4 와 같은 방향) |

**P1 이 이 실험의 전부다.** 무너지면 형태 축의 헤드라인은 직전 T 이야기고,
살아남으면 P 창 주장이 처음으로 **내용물 수준에서** 방어된다.

### 판정 조합 — 미리 박아둔다

- **P1 우월 ✅ · P2 미결/역전 · P3 ❌** → **P 창 신호는 진짜다.** 침입으로 설명 안 된다
- **P1 붕괴 · P2 붕괴 · P3 ✅** → **직전 T 침입이었다.** Q7-N 헤드라인을 소급 정정하고
  형태 축 문장을 P 창에서 떼어낸다
- **P1 이 「측정 불가」**(SAFE 코호트 검정력 부족) → ⛔ **어떤 결론 분기도 안 탄다**(R29 ②).
  필요 개체 수를 계산해 출력하고, 그 사실 자체를 결과로 쓴다(R30 ①)
- 갈리면 **갈렸다는 사실을 결과로 쓴다**(R18)

### 이 설계가 **미리 인정하는** 한계

1. **직전 T 위치는 모형 어림이다**(0.15~0.55 RR · 실측 T 주석이 없다). 그래서
   `T_HI ∈ {0.40, 0.45, 0.50}` × `여유 ∈ {0, 18}` **민감도 격자**를 함께 낸다(R28 ① ③).
2. **마스크가 라벨과 상관된다** — S 는 조기라 더 많이 잘린다. **선택 편향**이고,
   남은 S 비율·개체 수를 관문보다 **먼저** 출력한다(R17). 조용히 적은 n 으로 판정하지 않는다.
3. **검정력을 산다.** `pre ≥ 262샘플` 컷은 코호트를 크게 줄인다 — 그래서 **등가 프레임을
   기본으로 두지 않고**(R31 ①) 우월성 + 효과크기로 묻고, 필요 표본을 **모든 관문에서** 낸다.
4. **주 분석은 층화다**(R31 ③ ⑤ · P8) — 잔차 AUROC 는 팔 간 비교가 안 되고, 통제 강도별
   단조 감쇠에서 **가장 강한 통제**를 주 추정치로 써야 한다. 잔차화는 **감쇠 사다리**로만 쓴다.

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def equiv(lo, hi, margin):
    """**등가성 판정**(R29 ①). 미결을 등가로 읽지 않기 위해 별도 함수로 둔다.
    CI **전체**가 ±margin 안에 들어가야 「등가」다."""
    if lo > -margin and hi < margin:
        return "✅ 등가"
    if lo > margin or hi < -margin:
        return "❌ 차이 있음"
    return "⚠️ 미결"

def need_n(n, lo, hi, mean, margin):
    """★ 등가 판정에 **필요한 개체 수**(R30 ① · R31 ①). `n × (반폭 / (여유 − |점추정|))²`.

    점추정이 이미 여유 밖이면 표본을 아무리 늘려도 CI 가 여유 안에 못 들어간다 →
    **`None`(원리적으로 불가능)**. Q7-N 은 이걸 계산 안 하고 「미결」만 찍었다."""
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    slack = margin - abs(mean)
    if slack <= 0:
        return None
    return float(n) * (((hi - lo) / 2.0) / slack) ** 2

def judge(mean, lo, hi, n, margin):
    """★★ **등가와 우월성을 한 관문에서 둘 다** 판정한다(R31 ①).

    Q7-N 은 `p_full − stt` 에서 등가 「미결」만 찍고, **Bonferroni CI 하한이 +0.0387 로
    0 을 떼는데도** 요약에 「등가도 우월도 아니다」라고 적었다. 두 검정은 **다른 물음**이다.
    반환: (등가판정, 우월성판정, 필요개체, 프레임 문구)."""
    eq = equiv(lo, hi, margin)
    sup = decide(lo, hi, 0.0, ">")
    nn = need_n(n, lo, hi, mean, margin)
    if nn is None:
        frame = (f"⛔ **등가 판정 불가** — 점추정 {mean:+.4f} 이 여유 ±{margin} 밖이라 "
                 "어떤 표본으로도 불가능 → **우월성 + 효과크기 프레임**(R31 ①)")
    elif not np.isfinite(nn):
        frame = "⚠️ 필요 개체 계산 불가(CI 가 유한하지 않다)"
    else:
        frame = f"등가에 필요한 개체 ≈ **{nn:.0f}** (현재 {n})"
    return eq, sup, nn, frame

def slope_read(vals, tol=1e-9):
    """★ 통제 강도(약함→강함)별 같은 양의 사다리를 **부호가 아니라 기울기로** 읽는다(R31 ③).
    Q7-N 은 단조 감쇠(.0745→.0359)를 「네 기저 부호 일치 → 강건」으로 읽었다."""
    v = [x for x in vals if np.isfinite(x)]
    if len(v) < 3:
        return "⚠️ 사다리가 짧다"
    d = np.diff(np.abs(v))
    if np.all(d <= tol) and abs(v[0]) > 0 and abs(v[-1]) <= 0.7 * abs(v[0]):
        return "⚠️ **단조 감쇠 → 잔여 교란**. 가장 강한 통제를 주 추정치로 쓴다"
    if np.all(d <= tol):
        return "▸ 약한 감쇠"
    if np.all(d >= -tol):
        return "▸ 통제를 조일수록 커진다 — 누수가 신호를 만든 게 아니다(R30 ②)"
    return "▸ 비단조"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, glob, importlib, time
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
IDX_S   = 1
RPRE    = 100                   # svdb_labels._RPRE — 비트 배열에서 R 위치
NB_BOOT = 4000

# ── 창 (R = index 100 · 360Hz) — Q7-F/K/M/N 승계. **여기서 새로 고르지 않는다.**
SEGS = {"p_full": (0, 85), "p_early": (0, 32), "p_late": (53, 85),
        "stt": (130, 215), "stt_32": (130, 162), "stt_safe": (130, 175)}
WIDTH_PAIRS = (("p_late", "stt_32"), ("p_full", "stt"))   # (폭 32, 32) · (폭 85, 85)
INWIN_PAIR  = ("p_early", "p_late")                       # ★ P2 — 창 **안**을 쪼갠다(32, 32)

# ── ★★ 직전 T · 다음 QRS 의 기하 (이 노트북의 전부)
#    직전 T = [RPRE − T_LO·pre, RPRE − T_HI·pre]  (0.15~0.55 RR · Q7-F 승계)
#    ⚠️ **모형 어림이다** — 실측 T 주석이 없다. 그래서 아래 민감도 격자를 함께 낸다.
T_LO, T_HI = 0.85, 0.45
T_GUARD    = 18                      # 50ms 여유대 — 어림을 등호로 자르지 않는다
T_HI_SENS  = (0.40, 0.45, 0.50)      # ★ R28 ① ③ — 물려받은 상수를 파라미터로 감사
GUARD_SENS = (0, 18)

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
BASIS_K   = (4, 6, 8, 12, 16, 24, 32)   # 잔차화 기저(보조 · 감쇠 사다리용) — Q7-N 승계
PROBE_K   = (5, 10, 20)                 # ★ **기저에서 빼둔** 프로브
HIST_K    = 64
LB_K      = 16                          # 층화 두 번째 축 = f2_16 (Q7-M·N 승계)
F2_BIN    = 0.02                        # 층화 칸 폭 (Q7-N N6 승계)
TREND_W   = 8
MIN_S_TPL = 20                          # 전체 코호트 개체 채택 문턱
MIN_S_SAFE = 15                         # ★ SAFE 코호트 문턱 — 마스크가 표본을 산다
K_FOLD, K_FOLD_SAFE = 5, 3              # R22 — SAFE 는 작아서 겹을 줄인다(사전등록)
N_REPEAT  = 3
N_SHUF    = 20                          # R26 ② — null 의 셔플 오차를 CI 에 전파

# ★★ P8 — **주 분석은 층화다.** 잔차화는 감쇠 사다리(보조)로만 쓴다.
#    이유 둘: ① 잔차 AUROC 는 **팔 간 비교가 안 된다**(R31 ⑤ — lr_all 초과 +0.1424 <
#    stt +0.2144 라는 불가능한 순서가 나왔다) ② 통제 강도별 단조 감쇠에서는 **가장 강한
#    통제**를 주 추정치로 써야 하는데(R31 ③) 그게 층화다.
PRIMARY_METHOD = "strat"
LADDER = ("raw", "lin", "hist", "quad", "rank", "strat")   # 약한 통제 → 강한 통제

PROBE_MARGIN = 0.02          # 프로브 등가 여유
EQUIV_MARGIN = 0.05          # 관문 등가 여유
# ★ R30 ① — 여유를 거는 **모든** 관문에서 필요 표본을 계산한다. 사전 어림:
#   Q7-N 은 59개체에서 반폭 ≈0.038 이었다. SAFE 코호트는 개체가 줄어 반폭이 넓어지므로
#   **등가 판정은 애초에 어려울 것으로 예상**한다 → 우월성을 1차 프레임으로 둔다(R31 ①).
BONF3 = 0.05 / 3 / 2         # 1차 가족 {P1 · P2 · P3}
ISO_HI, ISO_LO = 0.7, 0.3

# ★★ 사전등록 규칙 체크리스트 (R29 ③) — 규칙을 아는 것과 적용하는 것은 다른 일이다.
RULE_CHECK = {
    "R16 fallback 없음":          "자산 셀에서 예외 삼킴 없음 · 목록 실패 시 중단",
    "R17 부분집합 산포":          "★ SAFE 코호트의 n·남은 S 비율을 관문보다 **먼저** 출력",
    "R22 교차적합":               "겹 밖 점수 · 반복 CV · SAFE 는 K_FOLD_SAFE",
    "R24-b ② 무너져야 할 대조군": "프로브(f2_5·10·20 · f1_rank)를 함께 채점",
    "R25 max 바닥 없음":          "개체별 max 바닥 없음 — 누출 바닥은 **팔별 매크로**의 max",
    "R26 ② 자기 null":            "라벨셔플 null 위 초과분 · SE 전파",
    "R27 ② 차이 우선":            "수준보다 (P 초과 − 음성대조 초과) 를 1차로",
    "R27 ③ 폭 정합":              "p_late(32)/stt_32(32) · p_full(85)/stt(85) · p_early/p_late(32)",
    "R28 ① 파라미터화":           "★ T_HI × 여유 민감도 격자로 마스크 정의를 감사",
    "R28 ② 하류 변수 금지":       "post_rr 은 기저에 **넣지 않는다**(마스크에만 쓴다)",
    "R28 ③ 물려받은 상수 감사":   "★ T_LO·T_HI 는 Q7-F 어림 — 이번에 민감도로 감사",
    "R29 ① 등가는 여유 사전등록": "★ ±0.05 · **모든 관문에서** 필요표본 계산(R30 ①)",
    "R29 ② 측정 불가 분기 금지":  "★ ⛔ 판정은 어떤 결론 분기도 타지 않는다",
    "R29 ④ 강건성도 CI":          "민감도 격자·층별을 CI 와 함께",
    "R30 ③ 폭은 축이다":          "폭-성능 곡선을 별도로 낸다",
    "R31 ① 등가/우월 둘 다":      "★ judge() 가 한 관문에서 둘 다 판정 + 필요표본",
    "R31 ② 누출 바닥 최댓값":     "★ 기저 밖 **모든 팔**(f4·f5 포함)의 max 로 깎는다",
    "R31 ③ 감쇠는 교란 신호":     "★ 사다리를 기울기로 읽는다 · 가장 강한 통제가 주 추정치",
    "R31 ④ 층 분해도 폭 정합":    "★ 층별을 WIDTH_PAIRS 로만",
    "R31 ⑤ 잔차는 팔 간 비교 X":  "★ 주 분석을 **층화**로 승격 · 잔차는 사다리에서만",
}

MASK_DOC = ("p_safe(seg): 직전 T 끝 RPRE−T_HI·pre ≤ seg시작−여유  |  "
            "q_safe(seg): 다음 QRS RPRE+post ≥ seg끝+여유  |  "
            "★ **두 마스크를 SEGS 의 모든 창에 걸어 하나의 SAFE 코호트**를 만든다 — "
            "한쪽 팔만 청소하면 비교가 깨진다")

CONFIG = dict(
    exp="quest46_q7p_p_safe", quest="ailab-2026-0046", step="svdb-p-safe",
    parent_exp=["quest46_q7n_residualize", "ailab-2026-0063"],
    purpose=("Q7-N 에서 두 결론이 동시에 나왔다 — 「폭을 맞추니 P 가 이긴다」(+0.0745)와 "
             "「P 창 안에서 이른 쪽이 더 세다」(p_early 0.2426 > p_late 0.2116). 둘을 "
             "동시에 참으로 만드는 유일한 설명은 **앞쪽 창이 담는 게 P 파가 아니라 직전 "
             "T** 라는 것이다. 침입은 **창의 내용물**이라 RR 특징 잔차화로는 안 지워진다. "
             "그래서 특징이 아니라 **표본**으로 친다 — 직전 T 가 P 창에 **기하학적으로 "
             "못 드는 비트만** 남기고 헤드라인을 다시 잰다. 살아남으면 P 주장이 처음으로 "
             "내용물 수준에서 방어되고, 무너지면 형태 축 헤드라인은 직전 T 이야기다"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    windows={k: list(v) for k, v in SEGS.items()},
    width_pairs=[list(p) for p in WIDTH_PAIRS], inwin_pair=list(INWIN_PAIR),
    t_model=dict(t_lo=T_LO, t_hi=T_HI, guard=T_GUARD,
                 t_hi_sens=list(T_HI_SENS), guard_sens=list(GUARD_SENS),
                 note="0.15~0.55 RR 모형 어림 — 실측 T 주석 없음(Q7-F 승계)"),
    mask_doc=MASK_DOC, primary_method=PRIMARY_METHOD, ladder=list(LADDER),
    basis_k=list(BASIS_K), probe_k=list(PROBE_K), hist_k=HIST_K, n_shuffle=N_SHUF,
    margins=dict(probe=PROBE_MARGIN, equiv=EQUIV_MARGIN),
    rule_check=RULE_CHECK,
    predictions={
        "P0": "(관문 아님 · **관문보다 먼저 출력**) SAFE 코호트의 개체 수 · 남은 S 비율 · "
              "침입률(S vs N). 마스크는 라벨과 상관되므로 **선택 편향**이다(R17)",
        "P1": f"★★ **주 관문** — SAFE 코호트 · 층화(주 분석) 에서 (p_full 초과) − "
              f"(stt 초과), 폭 85 정합. **등가와 우월성을 둘 다** 판정하고 필요 개체를 "
              f"계산한다(R31 ① · R30 ①). Bonferroni 3. 침입 가설이 맞으면 Q7-N 의 "
              f"+0.0745 가 **0 쪽으로 무너진다**",
        "P2": "★ **관문으로 승격** — SAFE 코호트에서 (p_early 초과) − (p_late 초과), "
              "둘 다 폭 32. Q7-N §4 는 점추정뿐인 **사후 관찰**이었다. 침입이 원인이면 "
              "SAFE 에서 **부호가 뒤집히거나 0 으로 붕괴**한다. Bonferroni 3",
        "P3": "★ ⚠️ **지지 = 교란 있음**(Q7-F F4 와 같은 방향) — rho(개체별 직전T 중첩 "
              "격차, 개체별 `p_early` 초과) CI 하한 > 0. Bonferroni 3",
        "P4": "(프레임) 등가가 원리적으로 불가능한 항목(점추정 > 여유)은 **우월성 + "
              "효과크기**로 읽는다. 필요 표본은 **모든 관문에서** 계산해 출력한다",
        "P5": "(관문 아님) 층 분해를 **폭 정합**으로 — 헤드라인이 런 5개체에 얹혀 "
              "있는지(R31 ④). Q7-N 층별 표는 STT(85) vs P_late(32) 라 분해가 안 됐다",
        "P6": "(관문 아님) 누출 바닥을 **기저 밖 모든 팔의 max**(f4·f5 포함)로 잡아 "
              "깎은 하한 병기(R31 ②). f2 프로브만 보면 3배 과소평가한다",
        "P7": "(관문 아님) **프로브에도 셔플 null 을 실측**한다. 이론상 0.5 지만 "
              "Q7-N 의 N2 판정은 **재지 않은 가정** 위에 서 있었다 — 싸니까 잰다",
        "P8": "(설계) 주 분석을 **층화로 승격**하고 잔차화는 **감쇠 사다리**로만 쓴다 "
              "(R31 ③ ⑤). 사다리는 raw → lin → hist → quad → rank → strat 순"},
    caveat=("★ **미리 인정하는 한계 넷**: ① **직전 T 위치는 모형 어림**(0.15~0.55 RR)이라 "
            "`T_HI` × 여유 민감도 격자를 함께 낸다 — 격자에서 결론이 갈리면 갈렸다고 쓴다 "
            "② **마스크가 라벨과 상관된다**(S 는 조기라 더 잘린다) → 선택 편향이고, 남은 "
            "S 비율·개체 수를 관문보다 먼저 낸다(R17) ③ **검정력을 산다** — `pre` 컷이 "
            "코호트를 줄이므로 등가 프레임을 기본으로 두지 않고 우월성으로 묻는다(R31 ①) "
            "④ 마스크는 **조기성 자체를 자른다** — SAFE 안에서는 S 의 조기성이 약해져 "
            "**모든 팔이 같이 내려갈 수 있다**. 그래서 수준이 아니라 **짝의 차이**를 "
            "1차로 읽는다(R27 ②). ★ 개체 내부 템플릿·로지스틱은 **라벨을 쓰는 상한**이지 "
            "배포 모형이 아니다. 학습 0회 · GPU 불필요 · 예상 40~70분(셔플이 지배)"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7p_p_safe", CONFIG, project=PROJECT)
run.log("설정 ✅ p_safe · 주 분석 = " + PRIMARY_METHOD + " · 사다리 " + " → ".join(LADDER))
run.log("  " + MASK_DOC)
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<26} {v_}")

In [ ]:
# CELL 2 — 【P-0a】 자산 · 매핑 (Q7-D~N 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)
BEAT = np.asarray(d5["beat"])[keep]
WIN = {k: np.ascontiguousarray(BEAT[:, :, a:b]).astype("float32")
       for k, (a, b) in SEGS.items()}
del BEAT
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【P-0a】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개")
for k_, (a_, b_) in SEGS.items():
    run.log(f"    {k_:<8} index {a_:>3}–{b_:<3} (폭 {b_-a_:>3})"
            f"  =  R{(a_-RPRE)/360*1000:+.0f}ms ~ R{(b_-RPRE)/360*1000:+.0f}ms")
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【P-A】 ★★ 직전 T 기하 · `p_safe` 마스크 · 특징 · 점수
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

# ─────────────────────────────────────────────────────────────────────────
# ★★ 이 노트북의 심장 — 창 안에 **무엇이 들어오는가**를 기하로 계산한다
# ─────────────────────────────────────────────────────────────────────────
def t_overlap(pre_v, seg, t_hi=T_HI, t_lo=T_LO):
    """직전 T 가 구간 `seg` 를 덮는 **비율**. ★ 0.15~0.55 RR **모형 어림**(Q7-F 승계).
    직전 T = [RPRE − t_lo·pre, RPRE − t_hi·pre] (현재 R 기준 index)."""
    s0, s1 = seg
    lo = RPRE - t_lo * np.asarray(pre_v, float)
    hi = RPRE - t_hi * np.asarray(pre_v, float)
    ov = np.clip(np.minimum(hi, s1) - np.maximum(lo, s0), 0, None)
    return ov / max(s1 - s0, 1)

def p_safe_seg(pre_v, seg, t_hi=T_HI, guard=T_GUARD):
    """★ 직전 T 가 `seg` 를 **물 수 없는** 비트. `RPRE − t_hi·pre ≤ seg시작 − 여유`.
    여유를 두는 이유: T 위치가 어림이라 등호로 자르면 어림 오차가 그대로 샌다."""
    return (RPRE - t_hi * np.asarray(pre_v, float)) <= (seg[0] - guard)

def q_safe_seg(post_v, seg, guard=T_GUARD):
    """★ 다음 QRS 가 `seg` 를 **물 수 없는** 비트. `RPRE + post ≥ seg끝 + 여유`.
    Q7-K 의 `stt_safe`(창을 줄이는 방식)를 **표본 쪽으로** 옮긴 판본이다."""
    return (RPRE + np.asarray(post_v, float)) >= (seg[1] + guard)

def safe_mask(pre_v, post_v, t_hi=T_HI, guard=T_GUARD, segs=None):
    """★★ **SEGS 의 모든 창에 두 마스크를 다 걸어** 하나의 코호트를 만든다.

    한쪽 팔만 청소하면 비교가 깨진다 — Q7-M 이 폭에서 저지른 실수(R27 ③)의 표본 판본이다.
    같은 비트 집합 위에서 모든 팔을 채점해야 짝의 차이를 읽을 수 있다(R31 ④)."""
    segs = SEGS if segs is None else segs
    pre_v = np.asarray(pre_v, float); post_v = np.asarray(post_v, float)
    m = np.ones(len(pre_v), bool)
    for s in segs.values():
        m &= p_safe_seg(pre_v, s, t_hi, guard) & q_safe_seg(post_v, s, guard)
    return m

def safe_threshold(t_hi=T_HI, guard=T_GUARD, segs=None):
    """마스크가 요구하는 **최소 pre / 최소 post**(샘플). 보고용."""
    segs = SEGS if segs is None else segs
    pre_min = max((RPRE - s[0] + guard) / t_hi for s in segs.values())
    post_min = max(s[1] + guard - RPRE for s in segs.values())
    return pre_min, post_min

# ─────────────────────────────────────────────────────────────────────────
# 특징 · 기저 (Q7-N 승계 — 여기서 새로 고르지 않는다)
# ─────────────────────────────────────────────────────────────────────────
def local_base(pre_v, k):
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    for i in range(n):
        a = max(0, i - k)
        out[i] = med if i - a < 3 else float(np.median(pre_v[a:i]))
    return out

def trend(pre_v, w):
    """직전 `w` 박동 RR 의 기울기 — 이력(rate hysteresis)의 대리. **선행 정보만** 쓴다."""
    n = len(pre_v); out = np.zeros(n); x = np.arange(w, dtype=float)
    xc = x - x.mean(); den = float((xc * xc).sum())
    for i in range(n):
        a = i - w
        if a < 0:
            continue
        y = pre_v[a:i]
        out[i] = float(((y - y.mean()) * xc).sum() / den)
    return out

def all_feats(pre_v, post_v):
    """기저 · 프로브 · 보고용 특징. **`post_rr` 은 기저에 안 들어간다**(R28 ②) —
    마스크(`q_safe`)에만 쓴다. 마스크는 통제가 아니라 **표본 정의**라 하류 변수 금지에
    걸리지 않는다: 라벨도 점수도 안 보고 **기하만** 본다."""
    med = float(np.median(pre_v)); n = len(pre_v)
    F = {}
    F["f1"] = med - pre_v
    for k in sorted(set(BASIS_K) | set(PROBE_K) | {HIST_K}):
        F[f"f2_{k}"] = 1.0 - pre_v / np.maximum(local_base(pre_v, k), 1e-9)
    b16 = local_base(pre_v, LB_K)
    b_first = local_base(pre_v, BASIS_K[0])
    F["f6"] = 1.0 - b16 / max(med, 1e-9)
    cv = np.empty(n)
    for i in range(n):
        a = max(0, i - TREND_W); w_ = pre_v[a:i] if i - a >= 3 else pre_v[:3]
        cv[i] = float(np.std(w_) / max(np.mean(w_), 1e-9))
    F["f4"] = cv
    F["trend"] = trend(pre_v, TREND_W)
    F["f5"] = np.r_[0.0, F[f"f2_{BASIS_K[0]}"][:-1]]
    F["f1_rank"] = stats.rankdata(F["f1"]) / n
    F["f3"] = 1.0 - (pre_v + post_v) / np.maximum(2.0 * b_first, 1e-9)
    F["f3_glob"] = 1.0 - (pre_v + post_v) / (2.0 * max(med, 1e-9))
    return F

def basis_mat(F, kind):
    """잔차화 기저(**보조** — 감쇠 사다리용). 열을 z-표준화한다(조건수)."""
    cols = ["f1"] + [f"f2_{k}" for k in BASIS_K] + ["f6"]
    Z = np.stack([F[c] for c in cols], axis=1)
    if kind == "quad":
        Z = np.c_[Z, Z ** 2]
    elif kind == "hist":
        Z = np.c_[Z, F["f4"], F["trend"], F[f"f2_{HIST_K}"]]
    elif kind not in ("lin", "rank"):
        raise AssetError(f"기저 종류 {kind} 를 모른다")
    elif kind == "rank":
        Z = np.stack([stats.rankdata(F[c]) / len(F[c]) for c in cols], axis=1)
    Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-12)
    return np.c_[np.ones(len(Z)), Z]

def prep(s, kind):
    s = np.asarray(s, float)
    return stats.rankdata(s) / len(s) if kind == "rank" else s

RESID_EPS = 1e-9

def residualize(s, Z):
    """점수에서 기저의 **선형 성분**을 뺀다. 라벨을 안 쓰므로 누수가 없다.
    기저 안 변수는 잔차가 정의상 0 이라 **정확히 0 으로 접는다** → AUROC 0.5."""
    s = np.asarray(s, float)
    if not np.isfinite(s).all():
        return None
    beta, *_ = np.linalg.lstsq(Z, s, rcond=None)
    e = s - Z @ beta
    if float(np.std(e)) <= RESID_EPS * (float(np.std(s)) + 1e-12):
        return np.zeros_like(e)
    return e

def strat_key(F, mask=None):
    """★ **주 분석**의 층 — `f1` 정확값 × `f2_16` 을 F2_BIN 으로 자른 칸(Q7-N N6 승계).
    조기성을 **모수 가정 없이** 통제한다. 잔차화와 달리 팔 간 수준 비교가 된다(R31 ⑤)."""
    f1 = F["f1"] if mask is None else F["f1"][mask]
    f2 = F[f"f2_{LB_K}"] if mask is None else F[f"f2_{LB_K}"][mask]
    pre_key = np.unique(np.round(f1, 6), return_inverse=True)[1].astype(np.int64)
    f2b = np.floor(f2 / F2_BIN).astype(np.int64)
    return pre_key * (int(f2b.max() - f2b.min()) + 1) + (f2b - f2b.min())

def strat_auc(sc, tt, key, min_s=1, min_n=1):
    """칸 안에서만 S/N 쌍을 세는 층화 AUROC. 칸을 넘는 비교를 안 하므로 조기성이 통제된다."""
    sc = np.asarray(sc, float)
    if not np.isfinite(sc).all():
        return float("nan"), 0, 0.0
    uq, inv = np.unique(key, return_inverse=True)
    num = den = 0.0; ks_ = 0
    for j in range(len(uq)):
        m = inv == j
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if len(s_) < min_s or len(n_) < min_n:
            continue
        ks_ += len(s_)
        gt = float((s_[:, None] > n_[None, :]).sum())
        eq = float((s_[:, None] == n_[None, :]).sum())
        num += gt + 0.5 * eq; den += float(len(s_) * len(n_))
    return (num / den if den >= 1 else float("nan")), ks_, den

def dist(B, ref):
    d = B - ref[None]
    return np.sqrt((d * d).sum(axis=(1, 2)))

def cv_logit(X, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            mu = X[tr].mean(0); sd = X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def two_template_cv(B, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            medN = np.median(B[tr & ~tt], axis=0); medS = np.median(B[tr & tt], axis=0)
            sc[te] = dist(B[te], medN) - dist(B[te], medS)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def run_struct(t_):
    prev_ = np.r_[False, t_[:-1]]; next_ = np.r_[t_[1:], False]
    iso = t_ & ~prev_ & ~next_
    rl = mx = 0
    for v in t_:
        rl = rl + 1 if v else 0
        mx = max(mx, rl)
    return float(iso.sum() / max(t_.sum(), 1)), int(mx)

LR_COLS = ["f1", "f2_8", "f2_16", "f3", "f4", "f5", "f6"]

def build_scores(F, tt, MORPH, seed, K, sub=None):
    """라벨을 쓰는 팔만 여기서 만든다(템플릿 · 로지스틱). **셔플 null 도 같은 함수.**
    `sub` 가 있으면 그 부분집합 위에서 **다시 적합**한다 — SAFE 코호트는 다른 표본이다."""
    S = {}
    X = np.stack([F[c] for c in LR_COLS], axis=1)
    if sub is not None:
        X = X[sub]; MORPH = {k: v[sub] for k, v in MORPH.items()}
    arms = {"lr_all": X, "lr_norr": X[:, [LR_COLS.index("f3"), LR_COLS.index("f4")]],
            "lr_f1": X[:, [LR_COLS.index("f1")]]}
    for nm_, XX in arms.items():
        sc_ = cv_logit(XX, tt, K, seed, N_REPEAT)
        if sc_ is None:
            return None
        S[nm_] = sc_
    for nm_, B_ in MORPH.items():
        st = two_template_cv(B_, tt, K, seed, N_REPEAT)
        S[nm_] = st if st is not None else np.full(len(tt), np.nan)
    return S

run.log("\n" + "=" * 100)
run.log("【P-A】 직전 T 기하 · SAFE 마스크 · 특징 · 점수")
run.log("=" * 100)
PRE_MIN, POST_MIN = safe_threshold()
run.log(f"  마스크가 요구하는 최소 pre = **{PRE_MIN:.1f}샘플** "
        f"({PRE_MIN/360*1000:.0f}ms · {60/(PRE_MIN/360):.0f}bpm 이하) · 최소 post = "
        f"{POST_MIN:.0f}샘플 ({POST_MIN/360*1000:.0f}ms)")
run.log(f"  (지배하는 창 — p_full/p_early 시작 index 0 · stt 끝 index 215)")

# ── 전역 침입률 (P0 · 관문보다 먼저 — R17)
_tt_all = (Y == IDX_S)
run.log("\n  직전 T 중첩률 (모형 어림 · 전체 비트) — **P 창별**")
OVG = {}
for k_ in ("p_full", "p_early", "p_late"):
    ov = t_overlap(PRE, SEGS[k_])
    OVG[k_] = dict(s=float(ov[_tt_all].mean()), n=float(ov[~_tt_all].mean()))
    run.log(f"    {k_:<8} S {OVG[k_]['s']:.3f} · N {OVG[k_]['n']:.3f}"
            f"  **격차 {OVG[k_]['s']-OVG[k_]['n']:+.3f}**")
run.log("    → `p_early` 격차가 크고 `p_late` 격차가 0 에 가까우면 **침입 가설과 일치**")
CONFIG["t_overlap_global"] = OVG

T0 = time.time()
FEAT, META, SKIP = {}, {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if int(tt.sum()) < MIN_S_TPL or int((~tt).sum()) < MIN_S_TPL:
        SKIP.append((int(r), f"S {int(tt.sum())} · N {int((~tt).sum())}")); continue
    pre_v, post_v = PRE[mm], POST[mm]
    F = all_feats(pre_v, post_v)
    sm = safe_mask(pre_v, post_v)
    iso_f, mx_run = run_struct(tt)
    FEAT[int(r)] = F
    META[int(r)] = dict(tt=tt, pre=pre_v, post=post_v, safe=sm,
                        MORPH={k: WIN[k][mm] for k in SEGS},
                        ov_early=t_overlap(pre_v, SEGS["p_early"]),
                        ov_late=t_overlap(pre_v, SEGS["p_late"]),
                        pos=int(tt.sum()), prev=float(tt.mean()),
                        iso_frac=iso_f, max_run=mx_run, n=int(len(mm)))

# ── 코호트 둘. **SAFE 는 같은 비트 집합 위에서 모든 팔을 채점한다.**
COHORTS = ("all", "safe")
def coh_sub(r, coh):
    return None if coh == "all" else META[r]["safe"]

SCORES = {c: {} for c in COHORTS}
KEEP = {c: [] for c in COHORTS}
for r in sorted(META):
    for coh in COHORTS:
        sub = coh_sub(r, coh)
        tt_c = META[r]["tt"] if sub is None else META[r]["tt"][sub]
        K_ = K_FOLD if coh == "all" else K_FOLD_SAFE
        floor_ = MIN_S_TPL if coh == "all" else MIN_S_SAFE
        if int(tt_c.sum()) < floor_ or int((~tt_c).sum()) < floor_:
            continue
        S = build_scores(FEAT[r], tt_c, META[r]["MORPH"], SEED0, K_, sub=sub)
        if S is None:
            continue
        SCORES[coh][r] = S; KEEP[coh].append(r)
RS = {c: sorted(KEEP[c]) for c in COHORTS}

# ── ★ P0 — 선택 편향을 **관문보다 먼저** 낸다 (R17)
run.log(f"\n  【P0】 코호트 (관문보다 **먼저** — 마스크는 라벨과 상관된다 · R17)")
run.log(f"    전체 코호트 채점 **{len(RS['all'])}개체** · 제외 {len(SKIP)}개체 "
        f"· {time.time()-T0:.0f}초")
run.log(f"    SAFE 코호트 채점 **{len(RS['safe'])}개체** "
        f"(문턱 S≥{MIN_S_SAFE} · 겹 {K_FOLD_SAFE})")
_keepfrac_s, _keepfrac_n, _prev0, _prev1 = [], [], [], []
for r in sorted(META):
    tt = META[r]["tt"]; sm = META[r]["safe"]
    _keepfrac_s.append(float((tt & sm).sum() / max(tt.sum(), 1)))
    _keepfrac_n.append(float((~tt & sm).sum() / max((~tt).sum(), 1)))
    _prev0.append(float(tt.mean())); _prev1.append(float(tt[sm].mean()) if sm.any() else np.nan)
run.log(f"    남은 비율 — **S {np.nanmean(_keepfrac_s):.3f}** · N {np.nanmean(_keepfrac_n):.3f}"
        f"   ⚠️ S 가 더 많이 잘린다(조기라서) = **선택 편향**")
run.log(f"    유병률 — 전체 {np.nanmean(_prev0):.4f} → SAFE {np.nanmean(_prev1):.4f}")
run.log(f"    ▸ 이 컷은 **조기성 자체를 자른다** — SAFE 안에서는 모든 팔이 같이 내려갈 수")
run.log(f"      있다. 그래서 수준이 아니라 **짝의 차이**를 1차로 읽는다(R27 ②)")
CONFIG["cohort"] = dict(n_all=len(RS["all"]), n_safe=len(RS["safe"]),
                        keep_s=float(np.nanmean(_keepfrac_s)),
                        keep_n=float(np.nanmean(_keepfrac_n)),
                        prev_all=float(np.nanmean(_prev0)),
                        prev_safe=float(np.nanmean(_prev1)),
                        pre_min=float(PRE_MIN), post_min=float(POST_MIN))

PURE = ["f1", "f2_16", "f3", "f3_glob", "f4", "f5", "f6"]
PROBES = [f"f2_{k}" for k in PROBE_K] + ["f1_rank"]
MORPH_ARMS = list(SEGS)
LR_ARMS = ["lr_all", "lr_norr", "lr_f1"]
ARMS = PURE + PROBES + MORPH_ARMS + LR_ARMS
LABEL_ARMS = MORPH_ARMS + LR_ARMS          # 재적합이 필요한 팔

def score_of(r, a, coh):
    """SAFE 코호트에서는 **부분집합 길이**의 점수를 돌려준다(팔마다 같은 비트 집합)."""
    if a in SCORES[coh][r]:
        return SCORES[coh][r][a]
    sub = coh_sub(r, coh)
    v = FEAT[r][a]
    return v if sub is None else v[sub]

def tt_of(r, coh):
    sub = coh_sub(r, coh)
    return META[r]["tt"] if sub is None else META[r]["tt"][sub]

run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【P-B】 채점 — **층화(주 분석)** + 잔차화(보조 · 감쇠 사다리)
# ★ P8 / R31 ⑤ — 잔차 AUROC 는 **팔 간 비교가 안 된다**. 그래서 주 분석은 층화이고,
#   잔차화는 「통제를 조이면 값이 어떻게 움직이나」(R31 ③)를 보는 **사다리**로만 쓴다.
run.log("\n" + "=" * 100)
run.log("【P-B】 층화(주 분석) · 잔차화(보조) — 코호트 2 × 사다리 6")
run.log("=" * 100)
BASES = ("lin", "rank", "quad", "hist")
MET = {c: {m: {a: np.full(len(RS[c]), np.nan) for a in ARMS} for m in LADDER}
       for c in COHORTS}
SKEEP = {c: np.zeros(len(RS[c]), bool) for c in COHORTS}
SFRAC = {c: np.full(len(RS[c]), np.nan) for c in COHORTS}
T0 = time.time()
for c in COHORTS:
    for i, r in enumerate(RS[c]):
        sub = coh_sub(r, c); tt = tt_of(r, c)
        Fs = {k: (v if sub is None else v[sub]) for k, v in FEAT[r].items()}
        key = strat_key(FEAT[r], sub)
        Z = {b: basis_mat(Fs, b) for b in BASES}
        v0, ks0, den0 = strat_auc(score_of(r, MORPH_ARMS[0], c), tt, key)
        if den0 >= 1:
            SKEEP[c][i] = True
            SFRAC[c][i] = ks0 / max(int(tt.sum()), 1)
        for a in ARMS:
            s = score_of(r, a, c)
            if not np.isfinite(s).all():
                continue
            MET[c]["raw"][a][i] = roc_auc_score(tt.astype(int), s)
            for b in BASES:
                e = residualize(prep(s, b), Z[b])
                if e is not None:
                    MET[c][b][a][i] = roc_auc_score(tt.astype(int), e)
            MET[c]["strat"][a][i] = strat_auc(s, tt, key)[0]
run.log(f"  ({time.time()-T0:.0f}초)")
for c in COHORTS:
    run.log(f"\n  [{c}] 층화 가능 **{int(SKEEP[c].sum())}/{len(RS[c])}** 개체 · "
            f"남은 S 중앙 {np.nanmedian(SFRAC[c][SKEEP[c]]) if SKEEP[c].any() else float('nan'):.3f}")
    run.log("    팔별 — " + " → ".join(LADDER))
    for a in ARMS:
        star = "  ★" if a in PROBES else ("  ▸" if a in MORPH_ARMS else "")
        run.log(f"      {a:<9} " + " · ".join(
            f"{np.nanmean(MET[c][m][a]):.4f}" for m in LADDER) + star)
run.log("\n  ▸ 잔차화 열의 **팔 간 수준 비교는 하지 않는다**(R31 ⑤) — 같은 팔의 코호트 간")
run.log("    비교와 **사전등록한 짝의 차이**만 읽는다. 주 지표는 `strat` 열이다")
CONFIG["metrics"] = {c: {m: {a: float(np.nanmean(MET[c][m][a])) for a in ARMS}
                         for m in LADDER} for c in COHORTS}
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【P-C】 라벨셔플 null — ★ **프로브·순수특징도 실측한다**(P7)
# Q7-N 은 순수 특징의 null 을 0.5 로 **가정**하고 N2 판정을 그 위에 세웠다. 이론상
# 치환 대칭으로 정확히 0.5 지만 — 층화 칸은 특징이 정하고 셔플은 라벨만 섞으므로
# 칸별 유병률이 흔들린다. **싸니까 잰다.** 재적합은 학습 낀 팔만 한다.
run.log("\n" + "=" * 100)
run.log(f"【P-C】 라벨셔플 null (셔플 {N_SHUF}회 · 코호트 2 · **전 팔 실측** · R26 ② · P7)")
run.log("=" * 100)
T1 = time.time()
NULL = {c: {m: {a: np.full(len(RS[c]), np.nan) for a in ARMS} for m in LADDER}
        for c in COHORTS}
NSE  = {c: {m: {a: np.full(len(RS[c]), np.nan) for a in ARMS} for m in LADDER}
        for c in COHORTS}
for c in COHORTS:
    K_ = K_FOLD if c == "all" else K_FOLD_SAFE
    for i, r in enumerate(RS[c]):
        sub = coh_sub(r, c); tt = tt_of(r, c)
        Fs = {k: (v if sub is None else v[sub]) for k, v in FEAT[r].items()}
        key = strat_key(FEAT[r], sub)
        Z = {b: basis_mat(Fs, b) for b in BASES}
        acc = {m: {a: [] for a in ARMS} for m in LADDER}
        for s_ in range(N_SHUF):
            rng = np.random.RandomState(SEED0 + 7919 * (s_ + 1) + int(r) + 13 * (c == "safe"))
            ts = rng.permutation(tt)                       # 유병률 보존
            Ss = build_scores(FEAT[r], ts, META[r]["MORPH"], SEED0 + 31 * (s_ + 1),
                              K_, sub=sub)
            if Ss is None:
                continue
            for a in ARMS:
                s = Ss[a] if a in Ss else Fs[a]
                if not np.isfinite(s).all():
                    continue
                acc["raw"][a].append(roc_auc_score(ts.astype(int), s))
                for b in BASES:
                    e = residualize(prep(s, b), Z[b])
                    if e is not None:
                        acc[b][a].append(roc_auc_score(ts.astype(int), e))
                acc["strat"][a].append(strat_auc(s, ts, key)[0])
        for m in LADDER:
            for a in ARMS:
                v_ = np.asarray([x for x in acc[m][a] if np.isfinite(x)], float)
                if len(v_) >= 2:
                    NULL[c][m][a][i] = float(v_.mean())
                    NSE[c][m][a][i] = float(v_.std(ddof=1) / np.sqrt(len(v_)))
run.log(f"  ({time.time()-T1:.0f}초)  **주 지표({PRIMARY_METHOD})** — 실측 vs null(±SE) vs 초과분")
for c in COHORTS:
    run.log(f"\n  [{c}]")
    for a in ARMS:
        m_ = np.nanmean(MET[c][PRIMARY_METHOD][a]); n_ = np.nanmean(NULL[c][PRIMARY_METHOD][a])
        se_ = np.nanmean(NSE[c][PRIMARY_METHOD][a])
        star = "  ★" if a in PROBES else ("  ▸" if a in MORPH_ARMS else "")
        run.log(f"    {a:<9} {m_:.4f}  null {n_:.4f} ±{se_:.4f}  **초과 {m_-n_:+.4f}**{star}")
run.log("\n  ▸ 순수 특징의 null 이 0.5 에서 유의하게 벗어나면 **가정이 틀린 것**이고,")
run.log("    Q7-N 의 N2 판정을 그만큼 다시 읽어야 한다 — 그래서 잰다(P7)")
CONFIG["null"] = {c: {a: float(np.nanmean(NULL[c][PRIMARY_METHOD][a])) for a in ARMS}
                  for c in COHORTS}
CONFIG["null_pure_dev"] = {c: float(np.nanmax([abs(np.nanmean(NULL[c][PRIMARY_METHOD][a]) - 0.5)
                                               for a in PURE + PROBES])) for c in COHORTS}
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【P-D】 관문 P1 · P2 · P3 (+ 필요표본 · 누출 바닥 · 감쇠 사다리)
def boot_pair(a1, a2, coh, meth, seed, nb=NB_BOOT, q=2.5):
    """짝의 **초과분 차이**를 개체 부트스트랩 + null 오차 전파(R26 ②)."""
    d = ((MET[coh][meth][a1] - NULL[coh][meth][a1])
         - (MET[coh][meth][a2] - NULL[coh][meth][a2]))
    se = np.sqrt(np.nan_to_num(NSE[coh][meth][a1]) ** 2
                 + np.nan_to_num(NSE[coh][meth][a2]) ** 2)
    ok = np.isfinite(d); d = d[ok]; se = se[ok]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.empty(nb)
    for b in range(nb):
        ix = rng.randint(0, len(d), len(d))
        v[b] = (d[ix] - rng.normal(0.0, 1.0, len(ix)) * se[ix]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

def boot_excess(a, coh, meth, seed, nb=NB_BOOT, q=2.5, const=None):
    d = MET[coh][meth][a] - (NULL[coh][meth][a] if const is None else const)
    se = np.nan_to_num(NSE[coh][meth][a]) if const is None else np.zeros(len(d))
    ok = np.isfinite(d); d = d[ok]; se = se[ok]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.empty(nb)
    for b in range(nb):
        ix = rng.randint(0, len(d), len(d))
        v[b] = (d[ix] - rng.normal(0.0, 1.0, len(ix)) * se[ix]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

def boot_rho(x, y, seed, nb=NB_BOOT, q=2.5):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 5:
        return float("nan"), float("nan"), float("nan"), 0
    r0 = float(stats.spearmanr(x, y).statistic)
    rng = np.random.RandomState(seed); v = []
    for _ in range(nb // 4):
        j = rng.randint(0, len(x), len(x))
        if len(np.unique(x[j])) > 2 and len(np.unique(y[j])) > 2:
            v.append(stats.spearmanr(x[j], y[j]).statistic)
    if len(v) < 20:
        return r0, float("nan"), float("nan"), len(x)
    return r0, float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(x)

run.log("\n" + "=" * 100)
run.log(f"【P-D】 관문 (주 분석 = {PRIMARY_METHOD} · SAFE {len(RS['safe'])}개체)")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

def gate_pair(gname, pa, pb, coh, note):
    """★ 등가와 우월성을 **둘 다** 판정하고 필요 개체를 **항상** 낸다(R31 ① · R30 ①)."""
    wa = SEGS[pa][1] - SEGS[pa][0]; wb = SEGS[pb][1] - SEGS[pb][0]
    if wa != wb:
        raise AssetError(f"폭 불일치 {pa}({wa}) vs {pb}({wb}) — R27 ③ 위반")
    m_, lo_, hi_, n_ = boot_pair(pa, pb, coh, PRIMARY_METHOD, SEED0 + 11, q=BONF3 * 100)
    if n_ < 3:
        DIFF[gname] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, pair=[pa, pb], width=wa,
                           coh=coh, need_n=None, equiv="⛔", sup="⛔")
        g_(gname, "⛔ 측정 불가", f"[{coh}] {pa}−{pb} — 개체 {n_} 로 CI 를 못 낸다")
        return
    eq, sup, nn, frame = judge(m_, lo_, hi_, n_, EQUIV_MARGIN)
    DIFF[gname] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, pair=[pa, pb], width=wa, coh=coh,
                       need_n=(None if nn is None else float(nn)), equiv=eq, sup=sup)
    g_(gname, sup, f"★ [{coh}] **폭 {wa}** ({pa} 초과) − ({pb} 초과) **{m_:+.4f}** "
                   f"[{lo_:+.4f}, {hi_:+.4f}] · Bonf 3 · n={n_}  {note}")
    run.log(f"       등가 {eq} · {frame}")
    su, slo, shi, _ = boot_pair(pa, pb, coh, PRIMARY_METHOD, SEED0 + 12)
    run.log(f"       (참고·미보정) {su:+.4f} [{slo:+.4f}, {shi:+.4f}] "
            f"· 우월성 {decide(slo, shi, 0.0, '>')}")
    # ★ 감쇠 사다리 — **부호가 아니라 기울기**로 읽는다(R31 ③)
    lad = []
    for meth in LADDER:
        lm, ll, lh, _ = boot_pair(pa, pb, coh, meth, SEED0 + 13)
        lad.append(lm)
        run.log(f"         [{meth:<5}] {lm:+.4f} [{ll:+.4f}, {lh:+.4f}]")
    run.log(f"       사다리 판정 — {slope_read(lad)}")
    DIFF[gname]["ladder"] = {m: float(v) for m, v in zip(LADDER, lad)}

# ── P1 ★★ 주 관문 — SAFE 코호트에서 폭 85 헤드라인이 살아남나
run.log("\n  P1 ★★ **주 관문** — 직전 T 가 못 드는 비트만 남기고 헤드라인 재계산")
gate_pair("P1", "p_full", "stt", "safe", "← Q7-N(all·lin) 은 +0.0745 였다")
run.log("\n    (대조) 같은 짝을 **전체 코호트**에서 — 마스크의 효과를 읽는 기준선")
gate_pair("P1r", "p_full", "stt", "all", "← 참고(Bonferroni 가족 밖)")

# ── P2 ★ 창 안을 쪼갠다 (둘 다 폭 32)
run.log("\n  P2 ★ **창 안** — `p_early` 가 `p_late` 를 이기는 게 침입 때문인가")
gate_pair("P2", INWIN_PAIR[0], INWIN_PAIR[1], "safe", "← Q7-N(all) 은 +0.0310 관찰")
run.log("\n    (대조) 전체 코호트")
gate_pair("P2r", INWIN_PAIR[0], INWIN_PAIR[1], "all", "← 참고(Bonferroni 가족 밖)")

# ── P3 ★ ⚠️ 지지 = 교란 있음
run.log("\n  P3 ★ ⚠️ **지지 = 교란 있음** (Q7-F F4 와 같은 방향)")
ov_gap, pe_ex, pl_ex = [], [], []
for i, r in enumerate(RS["all"]):
    tt = META[r]["tt"]
    ov_gap.append(float(META[r]["ov_early"][tt].mean() - META[r]["ov_early"][~tt].mean()))
    pe_ex.append(MET["all"][PRIMARY_METHOD]["p_early"][i]
                 - NULL["all"][PRIMARY_METHOD]["p_early"][i])
    pl_ex.append(MET["all"][PRIMARY_METHOD]["p_late"][i]
                 - NULL["all"][PRIMARY_METHOD]["p_late"][i])
r3, l3, h3, n3_ = boot_rho(ov_gap, pe_ex, SEED0 + 21)
DIFF["P3"] = dict(rho=r3, lo=l3, hi=h3, n=n3_)
g_("P3", decide(l3, h3, 0.0, ">"),
   f"rho(직전T 중첩격차, `p_early` 초과) **{r3:+.3f}** [{l3:+.3f}, {h3:+.3f}] · Bonf 3 "
   f"· n={n3_}   ⚠️ **지지 = 교란 있음**")
r3b, l3b, h3b, _ = boot_rho(ov_gap, pl_ex, SEED0 + 22)
run.log(f"       (대조) rho(중첩격차, **`p_late`** 초과) {r3b:+.3f} [{l3b:+.3f}, {h3b:+.3f}]"
        "  ← T 가 안 드는 자리라 상관이 약해야 한다")

# ── ★ 민감도 격자 — 마스크 정의(물려받은 어림)를 감사한다 (R28 ① ③ · R29 ④)
run.log("\n  민감도 격자 — `T_HI` × 여유 (⚠️ **null 없는 순수 층화 차이** · 미보정)")
run.log("    T 위치는 모형 어림이다. 격자에서 결론이 갈리면 **갈렸다고 쓴다**(R18)")
GRID = {}
for th in T_HI_SENS:
    for gd in GUARD_SENS:
        vals, keptn = [], 0
        for r in sorted(META):
            sm = safe_mask(META[r]["pre"], META[r]["post"], t_hi=th, guard=gd)
            tt = META[r]["tt"][sm]
            if int(tt.sum()) < MIN_S_SAFE or int((~tt).sum()) < MIN_S_SAFE:
                continue
            S = build_scores(FEAT[r], tt, META[r]["MORPH"], SEED0, K_FOLD_SAFE, sub=sm)
            if S is None:
                continue
            key = strat_key(FEAT[r], sm)
            va = strat_auc(S["p_full"], tt, key)[0]; vb = strat_auc(S["stt"], tt, key)[0]
            if np.isfinite(va) and np.isfinite(vb):
                vals.append(va - vb); keptn += 1
        pm, _ = safe_threshold(th, gd)
        GRID[f"th{th}_gd{gd}"] = dict(n=keptn, diff=float(np.mean(vals)) if vals else float("nan"),
                                      pre_min=float(pm))
        run.log(f"    T_HI {th:.2f} · 여유 {gd:>2} → pre ≥ {pm:6.1f}샘플 · 개체 {keptn:>2} · "
                f"p_full−stt {np.mean(vals) if vals else float('nan'):+.4f}")
CONFIG["sens_grid"] = GRID

# ── ★ 누출 바닥은 **기저 밖 모든 팔의 최댓값**으로 (R31 ②)
#    ⚠️ R25 와 헷갈리지 말 것: 여기 max 는 **개체별이 아니라 팔별 매크로**의 max 이고,
#    주장 값에서 **빼는** 쪽이라 방향이 보수적이다. R25 가 금한 건 개체별 max 로 부풀린
#    바닥을 문턱 0 과 비교해 **기각을 만드는 것**이었다.
run.log("\n  누출 바닥 (기저 밖에 남은 **모든 팔** — f2 프로브만 보면 3배 과소평가 · R31 ②)")
LEAK_ARMS = PROBES + ["f4", "f5", "f3", "f3_glob"]
fl = []
for a in LEAK_ARMS:
    mm_, ll_, hh_, _ = boot_excess(a, "safe", PRIMARY_METHOD, SEED0 + 15)
    fl.append(abs(mm_) if np.isfinite(mm_) else 0.0)
    run.log(f"    {a:<9} |초과| {abs(mm_) if np.isfinite(mm_) else float('nan'):.4f} "
            f"[{ll_:+.4f}, {hh_:+.4f}]")
LEAK_MAX = max(fl) if fl else 0.0
run.log(f"    ★ **보수적 누출 바닥 = {LEAK_MAX:.4f}**")
if np.isfinite(DIFF["P1"]["mean"]):
    run.log(f"       → P1 을 깎으면 {DIFF['P1']['mean']:+.4f} − {LEAK_MAX:.4f} = "
            f"**{DIFF['P1']['mean'] - LEAK_MAX:+.4f}**")
CONFIG["leak_max"] = float(LEAK_MAX)

# ── 폭-성능 곡선 (R30 ③ — 폭은 교란이 아니라 축이다)
run.log("\n  폭-성능 곡선 (주 지표 초과분 · 미보정 · R30 ③)")
for c in COHORTS:
    row = []
    for a in ("p_early", "p_late", "p_full", "stt_32", "stt_safe", "stt"):
        w = SEGS[a][1] - SEGS[a][0]
        mm_, _, _, _ = boot_excess(a, c, PRIMARY_METHOD, SEED0 + 16)
        row.append(f"{a}({w}) {mm_:+.4f}")
    run.log(f"    [{c}] " + " · ".join(row))
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【P-E】 P5 층 분해(★ **폭 정합**) · 침입률 · 강건성 (관문 아님)
run.log("\n" + "=" * 100)
run.log("【P-E】 P5 층 분해 · 침입률 (관문 아님)")
run.log("=" * 100)

# ── ★ 침입률 실측 (S vs N) — Q7-K L9 가 STT 에 했던 방식 그대로 P 에
run.log("  침입률 — 직전 T 가 P 창에 **실제로 드는 비트 비율**")
for k_ in ("p_full", "p_early", "p_late"):
    ins, inn = [], []
    for r in sorted(META):
        tt = META[r]["tt"]; ov = t_overlap(META[r]["pre"], SEGS[k_])
        ins.append(float((ov[tt] > 0).mean())); inn.append(float((ov[~tt] > 0).mean()))
    ins, inn = np.array(ins), np.array(inn)
    run.log(f"    {k_:<8} S {np.nanmean(ins):.4f} · N {np.nanmean(inn):.4f} · "
            f"**차이 {np.nanmean(ins - inn):+.4f}**")
    CONFIG.setdefault("intrusion", {})[k_] = dict(s=float(np.nanmean(ins)),
                                                  n=float(np.nanmean(inn)))

# ── ★ P5 층 분해 — **폭 정합 짝으로만**(R31 ④). Q7-N 은 STT(85) vs P_late(32) 였다
ISOF = {c: np.array([META[r]["iso_frac"] for r in RS[c]]) for c in COHORTS}
run.log("\n  P5 층 분해 — ★ **폭 정합 짝으로만**(R31 ④)")
for c in COHORTS:
    run.log(f"    [{c}] {len(RS[c])}개체")
    for nm_, msk in (("고립 S", ISOF[c] >= ISO_HI),
                     ("혼합", (ISOF[c] < ISO_HI) & (ISOF[c] > ISO_LO)),
                     ("런 우세", ISOF[c] <= ISO_LO)):
        n_ = int(msk.sum())
        if n_ < 3:
            run.log(f"      {nm_:<8} {n_}개체 — 3 미만이라 CI 를 내지 않는다(R17)"); continue
        parts = []
        for pa, pb in (WIDTH_PAIRS + (INWIN_PAIR,)):
            d = ((MET[c][PRIMARY_METHOD][pa] - NULL[c][PRIMARY_METHOD][pa])
                 - (MET[c][PRIMARY_METHOD][pb] - NULL[c][PRIMARY_METHOD][pb]))[msk]
            d = d[np.isfinite(d)]
            if len(d) < 3:
                parts.append(f"{pa}−{pb} n/a"); continue
            rng = np.random.RandomState(SEED0 + 22)
            v = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(800)]
            parts.append(f"{pa}−{pb}({SEGS[pa][1]-SEGS[pa][0]}) {d.mean():+.4f} "
                         f"[{np.percentile(v, 2.5):+.4f},{np.percentile(v, 97.5):+.4f}]")
            CONFIG.setdefault("layers", {}).setdefault(c, {}).setdefault(nm_, {})[
                f"{pa}-{pb}"] = float(d.mean())
        run.log(f"      {nm_:<8} {n_:>2}개체 · " + " · ".join(parts))
run.log("    ▸ 헤드라인이 **한 층(런 우세 소수 개체)에 얹혀 있으면** 그 사실을 결론에 쓴다")

# ── 순수 특징 null 이 정말 0.5 인가 (P7 의 결과를 읽는다)
run.log("\n  P7 — 순수 특징·프로브 null 의 0.5 이탈 (가정이 아니라 실측)")
for c in COHORTS:
    dev = [(a, float(np.nanmean(NULL[c][PRIMARY_METHOD][a]) - 0.5)) for a in PURE + PROBES]
    dev.sort(key=lambda t: -abs(t[1]))
    run.log(f"    [{c}] 최대 이탈 — " + " · ".join(f"{a} {v:+.4f}" for a, v in dev[:4]))
run.log("    ▸ 이탈이 프로브 여유(±%.2f)를 넘으면 **0.5 가정 위에 세운 판정을 다시 읽는다**"
        % PROBE_MARGIN)
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【P-F】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다(네모 방지).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))

# ① 직전 T 중첩 기하 — 왜 이 실험을 하는가
pre_grid = np.linspace(120, 480, 400)
for k_, c_ in (("p_full", "tab:blue"), ("p_early", "tab:red"), ("p_late", "tab:green")):
    ax[0].plot(pre_grid, t_overlap(pre_grid, SEGS[k_]), color=c_, label=k_)
ax[0].axvline(PRE_MIN, ls="--", lw=1.0, color="k")
ax[0].text(PRE_MIN, 0.9, f" safe cut {PRE_MIN:.0f}", fontsize=7)
ax[0].set_xlabel("pre_rr (samples @360Hz)"); ax[0].set_ylabel("prev-T overlap fraction")
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

# ② 사다리 — 통제를 조일수록 어떻게 움직이나
xs = np.arange(len(LADDER))
for gn, c_ in (("P1", "tab:blue"), ("P1r", "tab:gray"), ("P2", "tab:red")):
    lad = DIFF.get(gn, {}).get("ladder")
    if lad:
        ax[1].plot(xs, [lad[m] for m in LADDER], "o-", color=c_, label=gn)
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_xticks(xs); ax[1].set_xticklabels(LADDER, rotation=45, fontsize=7, ha="right")
ax[1].set_xlabel("control strength (weak to strong)")
ax[1].set_ylabel("width-matched pair difference")
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)

# ③ 관문 CI + 등가 여유대
gn_show = [g for g in ("P1", "P1r", "P2", "P2r") if np.isfinite(DIFF.get(g, {}).get("mean", np.nan))]
ys = np.arange(len(gn_show))
mm_ = [DIFF[g]["mean"] for g in gn_show]
lo_ = [DIFF[g]["lo"] for g in gn_show]; hi_ = [DIFF[g]["hi"] for g in gn_show]
if gn_show:
    ax[2].errorbar(mm_, ys, xerr=[np.array(mm_) - np.array(lo_),
                                  np.array(hi_) - np.array(mm_)],
                   fmt="o", color="tab:blue", capsize=4)
    ax[2].set_yticks(ys)
    ax[2].set_yticklabels([f"{g}: {DIFF[g]['pair'][0]}-{DIFF[g]['pair'][1]} "
                           f"[{DIFF[g]['coh']}]" for g in gn_show], fontsize=7)
ax[2].axvspan(-EQUIV_MARGIN, EQUIV_MARGIN, color="tab:green", alpha=.15)
ax[2].axvline(0, color="k", lw=.8)
ax[2].set_xlabel(f"pair difference  (shaded = equivalence +-{EQUIV_MARGIN})")
ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7p_p_safe", fig)
plt.close(fig)
display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("관문 요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⛔")     # ★ 측정 불가는 어떤 분기도 안 탄다(R29 ②)

for k in ("P1", "P2", "P3"):
    d_ = DIFF.get(k, {})
    extra = ""
    # ⛔ 관문에는 **등가·필요표본 주석도 붙이지 않는다** — 측정 못 한 값을 프레임 논의에
    # 얹으면 그게 곧 「측정 불가를 결론으로 쓰는 것」이다(R29 ②).
    if "equiv" in d_ and not un_(k):
        nn = d_.get("need_n")
        extra = (f"  · 등가 {d_['equiv']}"
                 + ("  · **등가 불가**(우월성 프레임)" if nn is None
                    else f"  · 등가 필요개체 ≈ {nn:.0f}" if np.isfinite(nn) else ""))
    run.log(f"  {k:<4}{VERD.get(k, '(미실행)')}{extra}")

run.log("")
if un_("P1"):
    run.log("  ⛔ **P1 측정 불가** — SAFE 코호트에 개체가 모자라 CI 를 못 냈다.")
    run.log("     **어떤 결론 분기도 타지 않는다**(R29 ②). 이 사실 자체가 결과다 —")
    run.log(f"     마스크가 요구한 pre ≥ {PRE_MIN:.0f}샘플 컷에서 S 가 "
            f"{CONFIG['cohort']['keep_s']:.3f} 만 남았다. 침입을 표본으로 치려면")
    run.log("     **SVDB 밖의 코호트(더 느린 기저 리듬)** 가 필요하다 — 그게 다음 설계다")
else:
    d1 = DIFF["P1"]; d1r = DIFF.get("P1r", {})
    run.log(f"  P1 — SAFE 에서 `p_full − stt` = **{d1['mean']:+.4f}** "
            f"[{d1['lo']:+.4f}, {d1['hi']:+.4f}]"
            + (f"  (전체 코호트 {d1r.get('mean', float('nan')):+.4f})" if d1r else ""))
    if ok_("P1"):
        run.log("  ★ **직전 T 가 못 드는 비트만 남겨도 P 창이 이긴다.**")
        run.log("     → 침입으로 설명되지 않는다. **P 창 주장이 처음으로 내용물 수준에서**")
        run.log("     방어됐다. 누출 바닥으로 깎은 하한도 함께 인용한다(R31 ②)")
        run.log(f"        깎은 하한 {d1['mean'] - LEAK_MAX:+.4f} (바닥 {LEAK_MAX:.4f})")
    elif no_("P1"):
        run.log("  ⛔ **P 창의 우위가 SAFE 에서 뒤집혔다** — 직전 T 침입이었다.")
        run.log("     Q7-N 헤드라인(+0.0745)을 **소급 정정**하고, 형태 축 문장에서")
        run.log("     P 창을 뗀다. 큐는 Q7-O(교차환자 템플릿) 대신 **STT 단독**으로 간다")
    else:
        run.log("  ⚠️ P1 미결 — 「없다」로 쓰지 않는다(R29 ①). 등가와 우월성은 다른 물음이고,")
        run.log("     둘 다 미결이면 **필요 개체 수**가 결과다(R30 ①)")
if un_("P2"):
    run.log("  ⛔ P2 측정 불가 — 결론으로 쓰지 않는다")
elif ok_("P2"):
    run.log("  ⚠️ P2 — SAFE 에서도 `p_early` 가 `p_late` 를 이긴다. 침입이 없는 자리에서도")
    run.log("     앞쪽이 세다면 **T 침입 말고 다른 앞쪽 성분**이 있다는 뜻이다")
elif no_("P2"):
    run.log("  ★ **P2 부호가 뒤집혔다** — 침입을 빼면 `p_late`(P 파 자리)가 이긴다.")
    run.log("     Q7-N §4 의 사후 관찰은 **침입이 만든 것**이었다")
else:
    run.log("  ⚠️ P2 미결 — Q7-N §4 의 사후 관찰을 확증도 반증도 못 했다")
if un_("P3"):
    run.log("  ⛔ P3 측정 불가 — 결론으로 쓰지 않는다")
elif ok_("P3"):
    run.log("  ⚠️ **P3 지지 = 교란이 있다** — 중첩 격차가 큰 개체일수록 `p_early` 가 잘 맞힌다.")
    run.log("     P1 이 살아남았더라도 이 상관은 **P 창 해석의 캐비앳**으로 함께 적는다")
else:
    run.log("  ▸ P3 — 개체 수준에서 중첩-성능 상관이 확인되지 않았다(교란의 직접 증거 없음)")
run.log("\n  ▸ 민감도 격자에서 결론이 갈렸는지 【P-D】 표를 확인한다 — 갈렸으면 갈렸다고 쓴다(R18)")
run.log("  ▸ 잔차 열의 팔 간 수준 비교는 하지 않았다(R31 ⑤). 주 지표는 층화다")

run.finish({
    "exp_id": "quest46_q7p_p_safe",
    "metric": "svdb_psafe_pfull_minus_stt",
    "value": float(DIFF.get("P1", {}).get("mean", float("nan"))),
    "passed": bool(ok_("P1")),
    "summary": ("직전 T 가 P 창에 기하학적으로 못 드는 비트만 남기고 폭 정합 헤드라인을 "
                "다시 쟀다. 주 분석은 층화이고, 등가와 우월성을 한 관문에서 둘 다 판정했다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "intrusion": CONFIG.get("intrusion", {}),
    "t_overlap_global": CONFIG.get("t_overlap_global", {}),
    "sens_grid": CONFIG.get("sens_grid", {}), "layers": CONFIG.get("layers", {}),
    "metrics": CONFIG.get("metrics", {}), "null": CONFIG.get("null", {}),
    "null_pure_dev": CONFIG.get("null_pure_dev", {}),
    "leak_max": CONFIG.get("leak_max"), "primary_method": PRIMARY_METHOD,
    "n_all": len(RS["all"]), "n_safe": len(RS["safe"]), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-p-safe`")